# Task 4: Fine-Tuning — NOVA Brand Voice Model

**NOVA AI Platform | Assessment Task 4**

This notebook fine-tunes **TinyLlama-1.1B-Chat** using **QLoRA** to always respond in NOVA's warm, empathetic, on-brand tone.

**Steps:**
1. Generate ~200 synthetic brand voice training pairs (via LLM)
2. Format dataset in ShareGPT/Alpaca format
3. Load TinyLlama with 4-bit quantization (QLoRA)
4. Configure LoRA adapters
5. Train with TRL `SFTTrainer` + W&B tracking
6. Evaluate brand voice quality
7. Save & push adapter to HuggingFace Hub

**Compute**: Google Colab Free Tier (T4 GPU) — ~15-20 min training

---

⚠️ **Important**: Enable GPU before running.
Runtime → Change runtime type → T4 GPU


## 1. Installation

In [ ]:
# Install all required packages
!pip install transformers==4.44.2 datasets peft==0.12.0 trl==0.10.1 bitsandbytes==0.43.3 \
    accelerate==0.34.2 wandb openai sentencepiece rouge-score nltk -q

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
import os
import json
import time
import random
import torch
import wandb

# ── API Keys (set via Colab Secrets) ─────────────────────────────────────────
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    HF_TOKEN = userdata.get('HF_TOKEN')
    WANDB_API_KEY = userdata.get('WANDB_API_KEY')
except Exception:
    OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')
    HF_TOKEN = os.getenv('HF_TOKEN', '')
    WANDB_API_KEY = os.getenv('WANDB_API_KEY', '')

# ── Model Config ──────────────────────────────────────────────────────────────
BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_DIR = './nova-brand-voice-adapter'
HF_REPO = f"{os.getenv('HF_USERNAME', 'your_username')}/nova-brand-voice-tinyllama"

# ── Training Config ───────────────────────────────────────────────────────────
TRAINING_CONFIG = {
    'num_train_epochs': 3,
    'per_device_train_batch_size': 4,
    'per_device_eval_batch_size': 4,
    'gradient_accumulation_steps': 4,  # Effective batch = 16
    'learning_rate': 2e-4,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.05,
    'max_seq_length': 512,
    'optim': 'paged_adamw_8bit',
    'logging_steps': 10,
    'eval_steps': 20,
    'save_steps': 50,
    'fp16': True,
    'report_to': 'wandb'
}

# ── QLoRA Config ──────────────────────────────────────────────────────────────
LORA_CONFIG = {
    'r': 16,
    'lora_alpha': 32,
    'target_modules': ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    'lora_dropout': 0.05,
    'bias': 'none',
    'task_type': 'CAUSAL_LM'
}

print('Configuration set:')
print(f'  Base model:    {BASE_MODEL}')
print(f'  Output dir:    {OUTPUT_DIR}')
print(f'  HF repo:       {HF_REPO}')
print(f'  Epochs:        {TRAINING_CONFIG["num_train_epochs"]}')
print(f'  Effective BS:  {TRAINING_CONFIG["per_device_train_batch_size"] * TRAINING_CONFIG["gradient_accumulation_steps"]}')
print(f'  LoRA rank:     {LORA_CONFIG["r"]}')

## 3. Generate Brand Voice Training Dataset

In [ ]:
# NOVA brand voice characteristics
NOVA_BRAND_VOICE = """NOVA brand voice:
- Warm, friendly, like a knowledgeable friend who works at NOVA
- Opens with empathy: 'We're so sorry', 'We completely understand'
- Uses inclusive 'we': 'we'd be happy', 'we'll sort this'
- Solution-forward: 'Here's exactly what we'll do'
- Closes with warmth: 'We're always here for you', 'You're in great hands'
- Uplifting even for bad news
- Never corporate: avoid 'as per our policy', 'your request has been received'
- Uses encouraging words: 'absolutely', 'of course', 'lovely'
- Short sentences. Clear structure."""

# Training examples by category (40 per category = 200 total)
# We'll generate these using the LLM
CATEGORY_PROMPTS = {
    'order_status': [
        'Where is my order ORD-1234? It was supposed to arrive yesterday.',
        'My tracking number shows no updates for 5 days.',
        'Has my order shipped yet? I placed it 3 days ago.',
        'I need my package by Friday, can you check delivery?',
        'The tracking link says delivered but I never received it.',
        'My order says processing for a week, is that normal?',
        'Can I upgrade to express shipping after placing my order?',
        'I moved. Can you update my delivery address?'
    ],
    'return_request': [
        'The dress I ordered is too small. How do I return it?',
        'I received the wrong item. I want a refund.',
        'My serum arrived broken. What should I do?',
        'I want to return my purchase. The quality is not what I expected.',
        'Can I exchange for a different size instead of a refund?',
        'I lost the receipt. Can I still return?',
        'How long does a refund take to appear in my account?',
        'The product caused a reaction. I need a refund urgently.'
    ],
    'product_query': [
        'Does your Vitamin C serum contain fragrance?',
        'Is the Glow Serum suitable for sensitive skin?',
        'I am allergic to lanolin. Which products should I avoid?',
        'Are your makeup products vegan?',
        'Does the SPF moisturizer also work as a primer?',
        'What is the shelf life after opening the serum?',
        'Can I layer the Vitamin C serum with retinol?',
        'Is the AHA exfoliant safe to use every day?'
    ],
    'sizing_query': [
        'I am usually a medium in other brands. Should I size up with NOVA?',
        'What is your EU 38 in US sizing for the ankle boots?',
        'I am between sizes. The blazer — should I go S or M?',
        'My measurements are: bust 88cm, waist 68cm. What size am I?',
        'Do your jeans run small or true to size?',
        'I have wide feet. Will your sneakers fit?',
        'How do I measure my foot for your footwear guide?',
        'I am a size 12 in the UK. What is that in your clothing?'
    ],
    'recommendation': [
        'I have oily skin and need a new moisturizer recommendation.',
        'I am looking for a gift for my mum. She likes skincare.',
        'I am building a beginner skincare routine. Where do I start?',
        'What are your best-selling makeup products?',
        'I want to try a new serum. What would you recommend for anti-aging?',
        'I have dry hair and need a hair mask recommendation.',
        'What accessories go well with a capsule wardrobe?',
        'I am trying to go vegan with my beauty routine. Recommendations?'
    ]
}

In [ ]:
from openai import OpenAI

llm = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url='https://openrouter.ai/api/v1',
    default_headers={'HTTP-Referer': 'https://github.com/nova-ai-platform', 'X-Title': 'NOVA Finetune'}
)

LLM_MODEL = 'mistralai/mistral-7b-instruct:free'

def generate_brand_voice_response(query: str, category: str) -> str:
    """Generate a NOVA brand voice response for a given query."""
    prompt = f"""You are writing training data for NOVA's AI support agent.

{NOVA_BRAND_VOICE}

Category: {category}
Customer query: {query}

Write a warm, on-brand NOVA response (3-5 sentences). Follow the brand voice exactly.
Include: empathy opener, clear action/answer, warm close.
Do NOT use generic phrases like 'as per our policy'.
Response:"""

    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.8,
        max_tokens=200
    )
    return response.choices[0].message.content.strip()


def generate_dataset(num_per_category: int = 40) -> list:
    """Generate full training dataset."""
    dataset = []

    for category, base_queries in CATEGORY_PROMPTS.items():
        print(f'\nGenerating {num_per_category} examples for category: {category}')

        # Use base queries + generate variations
        queries = base_queries.copy()

        # Generate more query variations if needed
        while len(queries) < num_per_category:
            base = random.choice(base_queries)
            variation_prompt = f"""Create a slightly different version of this customer support query:
Original: {base}
Variation (1 sentence, different wording):"""
            resp = llm.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': variation_prompt}],
                temperature=0.9, max_tokens=60
            )
            queries.append(resp.choices[0].message.content.strip())

        queries = queries[:num_per_category]

        # Generate responses for each query
        for i, query in enumerate(queries):
            try:
                response = generate_brand_voice_response(query, category)
                dataset.append({
                    'category': category,
                    'instruction': query,
                    'output': response
                })
                if (i + 1) % 10 == 0:
                    print(f'  {i+1}/{num_per_category} done')
            except Exception as e:
                print(f'  Error on {i}: {e}')
                continue

    print(f'\nDataset generated: {len(dataset)} examples')
    return dataset


# Generate dataset
training_data = generate_dataset(num_per_category=40)

In [ ]:
# Save training data
with open('nova_brand_voice_dataset.json', 'w') as f:
    json.dump(training_data, f, indent=2)

print(f'Saved {len(training_data)} training examples')
print('\nSample (first example):')
print(json.dumps(training_data[0], indent=2))

In [ ]:
# Format for TRL SFTTrainer (ChatML / TinyLlama chat template)
from datasets import Dataset as HFDataset

SYSTEM_MSG = """You are NOVA's AI support agent. Always respond in NOVA's warm, empathetic, on-brand tone.
Be solution-forward, use 'we', open with empathy, close with warmth."""

def format_for_training(example: dict) -> dict:
    """Format example in TinyLlama ChatML format."""
    formatted = f"""<|system|>
{SYSTEM_MSG}</s>
<|user|>
{example['instruction']}</s>
<|assistant|>
{example['output']}</s>"""
    return {'text': formatted, 'category': example['category']}

formatted_data = [format_for_training(ex) for ex in training_data]

# Train/eval split (90/10)
random.shuffle(formatted_data)
split_idx = int(len(formatted_data) * 0.9)
train_data = formatted_data[:split_idx]
eval_data = formatted_data[split_idx:]

train_dataset = HFDataset.from_list(train_data)
eval_dataset = HFDataset.from_list(eval_data)

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')
print('\nSample formatted text:')
print(train_dataset[0]['text'][:300])

## 4. Load Model with QLoRA (4-bit Quantization)

In [ ]:
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ── 4-bit Quantization Config (QLoRA) ────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',           # NF4 quantization (best quality)
    bnb_4bit_compute_dtype=torch.bfloat16,  # bf16 compute for speed
    bnb_4bit_use_double_quant=True        # Double quantization saves more memory
)

print(f'Loading {BASE_MODEL} with 4-bit quantization...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # Required for SFTTrainer

print(f'Model loaded. Parameters: {model.num_parameters():,}')

# Memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f'GPU memory used: {allocated:.2f} GB')

In [ ]:
# ── Prepare model for k-bit training ──────────────────────────────────────────
model = prepare_model_for_kbit_training(model)

# ── LoRA Configuration ────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_CONFIG['r'],
    lora_alpha=LORA_CONFIG['lora_alpha'],
    target_modules=LORA_CONFIG['target_modules'],
    lora_dropout=LORA_CONFIG['lora_dropout'],
    bias=LORA_CONFIG['bias'],
    task_type=LORA_CONFIG['task_type']
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Sanity check: trainable params should be ~0.5-2% of total
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 5. Initialize W&B Tracking

In [ ]:
# Initialize Weights & Biases
wandb.login(key=WANDB_API_KEY)

run = wandb.init(
    project='nova-brand-voice',
    name=f'tinyllama-qlora-{int(time.time())}',
    config={
        'base_model': BASE_MODEL,
        'lora_rank': LORA_CONFIG['r'],
        'lora_alpha': LORA_CONFIG['lora_alpha'],
        'learning_rate': TRAINING_CONFIG['learning_rate'],
        'epochs': TRAINING_CONFIG['num_train_epochs'],
        'effective_batch_size': (
            TRAINING_CONFIG['per_device_train_batch_size'] *
            TRAINING_CONFIG['gradient_accumulation_steps']
        ),
        'train_samples': len(train_dataset),
        'eval_samples': len(eval_dataset),
        'quantization': '4-bit NF4',
        'task': 'NOVA Brand Voice Fine-tuning'
    },
    tags=['qlora', 'brand-voice', 'tinyllama', 'nova']
)

print(f'W&B run initialized: {run.name}')
print(f'Dashboard: {run.url}')

## 6. Training

In [ ]:
# ── Training Arguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=TRAINING_CONFIG['num_train_epochs'],
    per_device_train_batch_size=TRAINING_CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size=TRAINING_CONFIG['per_device_eval_batch_size'],
    gradient_accumulation_steps=TRAINING_CONFIG['gradient_accumulation_steps'],
    learning_rate=TRAINING_CONFIG['learning_rate'],
    lr_scheduler_type=TRAINING_CONFIG['lr_scheduler_type'],
    warmup_ratio=TRAINING_CONFIG['warmup_ratio'],
    optim=TRAINING_CONFIG['optim'],
    logging_steps=TRAINING_CONFIG['logging_steps'],
    eval_steps=TRAINING_CONFIG['eval_steps'],
    evaluation_strategy='steps',
    save_steps=TRAINING_CONFIG['save_steps'],
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=TRAINING_CONFIG['fp16'],
    report_to=TRAINING_CONFIG['report_to'],
    run_name=run.name,
    group_by_length=True,  # Pack similar-length examples together (efficiency)
    dataloader_num_workers=2
)

# ── SFTTrainer ────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=TRAINING_CONFIG['max_seq_length'],
    args=training_args,
    packing=False  # Set True for shorter sequences (packs multiple into one)
)

print('Trainer configured. Starting training...')
print(f'Training samples: {len(train_dataset)}')
print(f'Eval samples: {len(eval_dataset)}')
print(f'Steps per epoch: {len(train_dataset) // (TRAINING_CONFIG["per_device_train_batch_size"] * TRAINING_CONFIG["gradient_accumulation_steps"])}')

In [ ]:
# ── Run Training ──────────────────────────────────────────────────────────────
train_start = time.time()

train_result = trainer.train()

train_time = time.time() - train_start
print(f'\nTraining complete in {train_time/60:.1f} minutes')
print(f'Final train loss: {train_result.training_loss:.4f}')

# Log final metrics to W&B
wandb.log({
    'train/final_loss': train_result.training_loss,
    'train/total_time_minutes': train_time / 60
})

## 7. Brand Voice Evaluation

In [ ]:
import re
from rouge_score import rouge_scorer
from transformers import pipeline

# Merge LoRA weights for inference
from peft import PeftModel

# Load merged model for inference
print('Loading model for inference...')
model.eval()

def generate_nova_response(query: str, max_new_tokens: int = 150) -> str:
    """Generate a response using the fine-tuned model."""
    prompt = f"""<|system|>
{SYSTEM_MSG}</s>
<|user|>
{query}</s>
<|assistant|>
"""
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant response
    if '<|assistant|>' in generated:
        generated = generated.split('<|assistant|>')[-1].strip()
    return generated


def score_brand_voice(response: str) -> dict:
    """Score a response on NOVA brand voice dimensions."""
    text_lower = response.lower()

    # 1. Empathy score: does it open with empathy?
    empathy_phrases = ['sorry', 'understand', 'completely', 'of course', 'absolutely',
                      'we\'re here', 'thank you']
    empathy = sum(1 for p in empathy_phrases if p in text_lower)
    empathy_score = min(empathy / 2, 1.0)  # Normalize 0-1

    # 2. Warmth score: uses 'we', warm closings
    warmth_phrases = [' we ', 'we\'ll', 'we\'re', 'our team', 'always here', 'happy to',
                     'lovely', 'wonderful', 'great news']
    warmth = sum(1 for p in warmth_phrases if p in text_lower)
    warmth_score = min(warmth / 3, 1.0)

    # 3. Action score: solution-forward
    action_phrases = ['here\'s', 'let me', 'we\'ll', 'i\'ve', 'right away',
                     'immediately', 'sorted', 'resolved']
    action = sum(1 for p in action_phrases if p in text_lower)
    action_score = min(action / 2, 1.0)

    # 4. Avoid corporate language (negative)
    corporate_phrases = ['as per', 'kindly', 'your request has been',
                        'please be informed', 'unable to', 'i am an ai']
    corporate = sum(1 for p in corporate_phrases if p in text_lower)
    corporate_penalty = min(corporate * 0.2, 0.4)

    # 5. Length check (not too short, not too long)
    word_count = len(response.split())
    length_score = 1.0 if 30 <= word_count <= 120 else 0.5

    overall = (empathy_score * 0.3 + warmth_score * 0.3 +
               action_score * 0.2 + length_score * 0.2 - corporate_penalty)
    overall = max(0.0, min(1.0, overall))

    return {
        'empathy_score': round(empathy_score, 2),
        'warmth_score': round(warmth_score, 2),
        'action_score': round(action_score, 2),
        'corporate_penalty': round(corporate_penalty, 2),
        'length_score': round(length_score, 2),
        'word_count': word_count,
        'overall_brand_voice_score': round(overall, 3)
    }

print('Evaluation functions ready')

In [ ]:
# Evaluate on 20 held-out test queries
TEST_QUERIES = [
    ('Where is my order ORD-5678? It is 5 days late.', 'order_status'),
    ('I want to return my boots. They hurt my feet.', 'return_request'),
    ('Does your Vitamin C serum contain fragrance?', 'product_query'),
    ('I am a UK size 14. What NOVA size is that?', 'sizing_query'),
    ('I have dry skin. What moisturizer do you recommend?', 'recommendation'),
    ('My package tracking hasn\'t updated in a week.', 'order_status'),
    ('I received the wrong color lipstick. Please help.', 'return_request'),
    ('Is the Retinol Night Repair safe for sensitive skin?', 'product_query'),
    ('What are your shoe sizes? I am usually EU 38.', 'sizing_query'),
    ('I want to start a vegan skincare routine. Where do I begin?', 'recommendation'),
    ('My delivery says arriving today but nothing yet.', 'order_status'),
    ('The serum I bought leaked in the packaging.', 'return_request'),
    ('Are your hair masks free from silicones?', 'product_query'),
    ('I am between size S and M. Which should I order?', 'sizing_query'),
    ('What skincare products are good for acne?', 'recommendation'),
    ('Can I still track my order if I lost the confirmation email?', 'order_status'),
    ('I bought this as a gift and they don\'t like it. Can we return?', 'return_request'),
    ('Can I use the AHA exfoliant with the niacinamide serum?', 'product_query'),
    ('My foot is 24.5cm. What shoe size should I order?', 'sizing_query'),
    ('What is your best anti-aging product?', 'recommendation')
]

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
all_scores = []

print('Running brand voice evaluation on 20 test queries...\n')
for i, (query, category) in enumerate(TEST_QUERIES):
    response = generate_nova_response(query)
    bv_scores = score_brand_voice(response)
    all_scores.append(bv_scores)

    print(f'[{i+1:02d}] {category:<20} | BV={bv_scores["overall_brand_voice_score"]:.2f} | '
          f'Words={bv_scores["word_count"]} | {query[:40]}...')

import pandas as pd
import numpy as np

scores_df = pd.DataFrame(all_scores)
print('\n=== BRAND VOICE EVALUATION SUMMARY ===')
print(scores_df.mean().round(3))

avg_bv = scores_df['overall_brand_voice_score'].mean()
print(f'\nMean Brand Voice Score: {avg_bv:.3f}/1.0')

# Log to W&B
wandb.log({'eval/brand_voice_score': avg_bv,
           'eval/empathy_score': scores_df['empathy_score'].mean(),
           'eval/warmth_score': scores_df['warmth_score'].mean()})

## 8. Save & Upload Model

In [ ]:
# Save adapter locally
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}')

In [ ]:
# Push to HuggingFace Hub
from huggingface_hub import login

login(token=HF_TOKEN)

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

print(f'Model pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# Save evaluation report
eval_report = {
    'task': 'Task 4 - Brand Voice Fine-tuning',
    'base_model': BASE_MODEL,
    'fine_tuned_model': HF_REPO,
    'training_config': TRAINING_CONFIG,
    'lora_config': LORA_CONFIG,
    'train_samples': len(train_dataset),
    'eval_samples': len(eval_dataset),
    'final_train_loss': float(train_result.training_loss),
    'training_time_minutes': round(train_time / 60, 1),
    'brand_voice_evaluation': {
        'num_test_queries': len(TEST_QUERIES),
        'mean_brand_voice_score': round(avg_bv, 4),
        'mean_empathy_score': round(float(scores_df['empathy_score'].mean()), 4),
        'mean_warmth_score': round(float(scores_df['warmth_score'].mean()), 4),
        'mean_action_score': round(float(scores_df['action_score'].mean()), 4),
    },
    'wandb_run': run.url if run else None
}

with open('task4_eval_report.json', 'w') as f:
    json.dump(eval_report, f, indent=2)

print('Evaluation report saved to task4_eval_report.json')
wandb.finish()
print(f'W&B run complete. Link: {run.url}')

## 9. Inference Demo — Before vs After Fine-tuning

Compare the base model's response vs the fine-tuned model's response.

In [ ]:
demo_queries = [
    'My order is late. What is happening?',
    'The mascara I bought smudges. I want a refund.',
    'What moisturizer do you recommend for dry skin?'
]

print('=== FINE-TUNED MODEL OUTPUT (NOVA Brand Voice) ===')
for query in demo_queries:
    print(f'\nQuery: {query}')
    response = generate_nova_response(query)
    scores = score_brand_voice(response)
    print(f'Response: {response}')
    print(f'Brand Voice Score: {scores["overall_brand_voice_score"]:.2f}')